# 00_project_scope_and_sources\n\n## Phase 1 Pilot Scope\n\nThis notebook documents the controlled source feasibility scope for Phase 1. It is not the final city reference model and does not implement ingestion logic.\n\n| city_name | country_code | latitude | longitude | reason |\n| --- | --- | ---: | ---: | --- |\n| Vienna | AT | 48.2082 | 16.3738 | Likely coverage across Open-Meteo, Wikipedia, and EEA. |\n| Berlin | DE | 52.5200 | 13.4050 | Likely coverage across Open-Meteo, Wikipedia, and EEA. |\n\nTarget pollutants for Phase 1 checks: PM2.5, PM10, NO2.\n\nIf a pollutant is unavailable or named differently in a source, record it as a source-specific constraint. Do not expand the pollutant scope in Phase 1.\n\nPhase 1 must use these same two pilot cities across the Open-Meteo, EEA, and Wikipedia source spikes.\n\n## Open-Meteo Phase 1 Feasibility Notes\n\nStatus: usable.\n\nA controlled one-day hourly Open-Meteo Air Quality API request was tested for Vienna and Berlin with `timezone=UTC` and hourly fields `pm10,pm2_5,nitrogen_dioxide`.\n\n| city_name | evidence file | observed hourly fields | records | missing pollutant values in sample |\n| --- | --- | --- | ---: | ---: |\n| Vienna | `data/bronze/open_meteo_raw/sample_open_meteo_vienna_at.json` | `time`, `pm10`, `pm2_5`, `nitrogen_dioxide` | 24 | 0 |\n| Berlin | `data/bronze/open_meteo_raw/sample_open_meteo_berlin_de.json` | `time`, `pm10`, `pm2_5`, `nitrogen_dioxide` | 24 | 0 |\n\nTimestamp assumption: requests used UTC and the response reported `utc_offset_seconds=0`. The hourly timestamp field is `hourly.time`.\n\nPollutant field mapping for later phases: PM2.5 maps to `pm2_5`, PM10 maps to `pm10`, and NO2 maps to `nitrogen_dioxide`.\n\nRisk: this source spike proves short-window reachability only. Later ingestion must still handle missing values and must not compare Open-Meteo forecast/current data to EEA historical data without context fields.\n\n## EEA Phase 1 Feasibility Notes\n\nStatus: usable with constraints.\n\nEEA availability was checked at metadata level through the station spatial service. This notebook does not download full EEA time series, implement ingestion, define station-radius matching, or run Spark.\n\n| city | metadata evidence | target pollutant evidence | key risk | decision |\n| --- | --- | --- | --- | --- |\n| Vienna | 37 nearby station records with target pollutant coverage in inspected metadata. | Examples: `Taborstraße` (`AT90TAB`) and `AKH` (`AT90AKC`) report NO2, PM2.5, and PM10. | Need documented station selection rule. | usable with constraints |\n| Berlin | 68 nearby station records with target pollutant coverage in inspected metadata. | Example: `Berlin Mitte` (`DEBE068`) reports NO2, PM10, and PM2.5 with different year coverage. | PM2.5 history and station representativeness need review. | usable with constraints |\n\nExpected future ingestion fields: station identifier, station name, timestamp or period, pollutant, value, unit, quality or validity indicator, and source metadata.\n\nMain Phase 2 risk: EEA is station-based, not city-based. City-level analysis requires an explicit station-to-city mapping rule before any EEA aggregation is built.\n\n## Wikipedia Phase 1 Feasibility Notes\n\nStatus: usable with constraints.\n\nWikipedia HTML was fetched for Vienna and Berlin. A small raw infobox HTML excerpt was saved for each pilot city under `data/bronze/wikipedia_html/`. This is source-spike evidence only, not a production scraper or parser implementation.\n\n| city | evidence file | infobox found | parseable candidates | decision |\n| --- | --- | --- | --- | --- |\n| Vienna | `data/bronze/wikipedia_html/sample_wikipedia_vienna_at.html` | yes | country, area, population, coordinates | usable with constraints |\n| Berlin | `data/bronze/wikipedia_html/sample_wikipedia_berlin_de.html` | yes | country, area, population, coordinates | usable with constraints |\n\nMain risk: Wikipedia infobox labels are not a stable API contract. Phase 4 must implement conservative parser tests and preserve raw HTML evidence.\n\nFallback strategy: if population, area, or coordinates are missing or ambiguous, keep the raw HTML sample, set parsed fields to null, and document the limitation instead of guessing.\n\n## Phase 1 Source Feasibility Matrix\n\n| Source | Status | Format | Key Fields | Risks | Decision |\n| --- | --- | --- | --- | --- | --- |\n| Open-Meteo Air Quality API | usable | JSON | `hourly.time`, `pm2_5`, `pm10`, `nitrogen_dioxide`, units, coordinates | field naming and missing values | use later for API/Kafka path after schema work |\n| EEA historical air quality data | usable with constraints | station metadata plus Parquet/download exports | station, pollutant, unit, year coverage, future timestamp/value fields | station-to-city mapping | proceed to Phase 2 mapping before ingestion |\n| Wikipedia city pages | usable with constraints | HTML infobox/page markup | country, population, area, coordinates | page structure changes | parse carefully in Phase 4 with raw HTML evidence |\n\nPhase 2 readiness: Go with constraints. The next phase must focus on the city reference model and station/source mapping. Do not skip directly into full ingestion, Kafka, Spark, or Gold-layer implementation.\n